In [49]:
import pandas as pd
import joblib
from tqdm import tqdm
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')
calibrated = joblib.load('lightgbm.pkl')

cols_to_drop = [ 
    'dischtime', 
    'current_date',
    'target_readmission_30d',
    'los'
]
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]

X = ind_hosp.drop(columns=cols_to_drop)
bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

def get_age_group(age):
    if age < 40:
        return "young"
    elif age < 65:
        return "middle"
    elif age < 80:
        return "elderly"
    else:
        return "very_elderly"

X = X.copy()
X["age_group"] = X["age"].apply(get_age_group)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']
train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

# sample_size = 1000
# test_patients_sample, _ = train_test_split(
#     test_ids,
#     train_size=sample_size,
#     random_state=42,
#     stratify=patient_target[patient_target['subject_id'].isin(test_ids)]['has_readmission']
# )
# test_patients = test_patients_sample.tolist()
test_patients = test_ids.tolist()

print(f"Test patients: {len(test_patients)}")

Test patients: 35915


In [3]:
reference_stats = (
    X
    .groupby(["gender_male", "age_group"])
    .agg(["median", "mean", lambda x: x.mode().iat[0]])
)

def get_reference(feature, patient, method):
    gender = patient["gender_male"]
    age_group = patient["age_group"]
    return reference_stats.loc[
        (gender, age_group),
        (feature, method)
    ]

reference_stats

subject_id                              hadm_id  \
                              median          mean <lambda_0>      median   
gender_male age_group                                                       
0           elderly       15012885.0  1.501095e+07   15114531  24936567.0   
            middle        15046509.0  1.502763e+07   13297743  24989011.0   
            very_elderly  15077863.0  1.504156e+07   17200404  24970508.0   
            young         15015994.0  1.500889e+07   11965254  24984615.0   
1           elderly       15004302.0  1.501130e+07   10108435  25020522.0   
            middle        14952873.0  1.498086e+07   12468016  25021409.0   
            very_elderly  14926277.0  1.497121e+07   14061397  25042674.0   
            young         15039520.0  1.500283e+07   17295976  25013003.0   

                                                     age             \
                                  mean <lambda_0> median       mean   
gender_male age_group                                                 
0           elderly       2.497827e+07   20142342   72.0  71.779658   
            middle        2.500125e+07   20212982   55.0  53.835389   
            very_elderly  2.498995e+07   20019675   86.0  86.273661   
            young         2.499282e+07   20372141   31.0  30.663121   
1           elderly       2.499684e+07   20186914   71.0  71.508746   
            middle        2.501278e+07   20197189   56.0  54.718089   
            very_elderly  2.502732e+07   20004811   85.0  85.570985   
            young         2.500768e+07   20058812   31.0  30.293755   

                                       day  ...  \
                         <lambda_0> median  ...   
gender_male age_group                       ...   
0           elderly              71    3.0  ...   
            middle               64    3.0  ...   
            very_elderly         91    3.0  ...   
            young                33    2.0  ...   
1           elderly              66    4.0  ...   
            middle               64    3.0  ...   
            very_elderly         91    4.0  ...   
            young                38    3.0  ...   

                         admission_location_PROCEDURE SITE  \
                                                <lambda_0>   
gender_male age_group                                        
0           elderly                                      0   
            middle                                       0   
            very_elderly                                 0   
            young                                        0   
1           elderly                                      0   
            middle                                       0   
            very_elderly                                 0   
            young                                        0   

                         admission_location_TRANSFER FROM HOSPITAL            \
                                                            median      mean   
gender_male age_group                                                          
0           elderly                                            0.0  0.172390   
            middle                                             0.0  0.129974   
            very_elderly                                       0.0  0.171995   
            young                                              0.0  0.075563   
1           elderly                                            0.0  0.192526   
            middle                                             0.0  0.161516   
            very_elderly                                       0.0  0.198167   
            young                                              0.0  0.123097   

                                     \
                         <lambda_0>   
gender_male age_group                 
0           elderly               0   
            middle                0   
            very_elderly          0   
            young                 0 

In [4]:
icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

features_to_analyze = [f for f in features_to_analyze if f in X.columns]
print(f"Analyzing {len(features_to_analyze)} features")

Analyzing 60 features


In [5]:
clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 145}, #sodium
    'lab_50971_daily': {'low': 3.5, 'high': 5}, #potassium
    'lab_50902_daily': {'low': 95, 'high': 105}, #chloride
    'lab_50882_daily': {'low': 22, 'high': 32}, #bicarbonate
    'lab_50912_daily': {'low': 0.8, 'high': 1.3}, #creatinine
    'lab_51006_daily': {'low': 8, 'high': 21}, #BUN
    'lab_50931_daily': {'low': 65, 'high': 110}, #glucose
    'lab_50893_daily': {'low': 8.5, 'high': 10.2}, #calcium
    'lab_50868_daily': {'low': 10, 'high': 18}, #anion gap
    'lab_51222_daily': {
        'male': {'low': 13, 'high': 17},
        'female': {'low': 12, 'high': 15}
    }, #hemoglobin
    'lab_51301_daily': {'low': 4, 'high': 10}, #WBC
    'lab_51265_daily': {'low': 150, 'high': 400}, #Platelet Count
    'lab_51221_daily': {
        'male': {'low': 40, 'high': 52},
        'female': {'low': 36, 'high': 47}
    }, #Hematocrit
    'lab_51250_daily': {'low': 80, 'high': 100}, #MCV
    'lab_51277_daily': {'low': 11.5, 'high': 14.5}, #RDW
    'lab_50960_daily': {'low': 1.5, 'high': 2}, #magnesium
    'lab_50970_daily': {'low': 2.7, 'high': 4.5}, #phosphate
    'lab_51248_daily': {'low': 26, 'high': 32}, #MCH
    'lab_51249_daily': {'low': 30, 'high': 35}, #MCHC
    'lab_51279_daily': {
        'male': {'low': 4.5, 'high': 5.9},
        'female': {'low': 4, 'high': 5.2}
    } #RBC
}

In [6]:
def get_counterfactual_value(feature, patient_value, patient_row):
    if feature.startswith('lab_'):
        ranges = clinical_ranges[feature]
        if 'male' in ranges:
            if patient_row['gender_male'] == 1:
                ref = ranges['male']
            else:
                ref = ranges['female']
        else:
            ref = ranges

        low = ref['low']
        high = ref['high']
        reference = (low + high) / 2
        
        if patient_value < low:
            direction = 'low'
            deviation = patient_value - low
        elif patient_value > high:
            direction = 'high'
            deviation = patient_value - high
        else:
            direction = 'normal'
            reference = patient_value
            deviation = 0

        return reference, direction, deviation
    
    if feature == 'age':
        age_norm = get_reference('age', patient_row, 'median')
        if patient_value < age_norm:
            return age_norm, 'younger', patient_value - age_norm
        else:
            return age_norm, 'older', patient_value - age_norm
    
    if feature == 'gender_male':
        return 1 if patient_value == 0 else 0, 'opposite', 0
    
    if feature in ['num_medications_daily', 'num_procedures_daily']:
        if patient_value > 0:
            return 0, 'remove', -patient_value
        else:
            return 1, 'add', 1
    
    if feature == 'cumulative_medications':
        mean_val = get_reference('cumulative_medications', patient_row, 'median')
        if patient_value > mean_val:
            return mean_val, 'reduce_to_median', patient_value - mean_val
        else:
            return mean_val, 'increase_to_mean', patient_value - mean_val
    
    if feature == 'cumulative_procedures':
        mean_val = get_reference('cumulative_procedures', patient_row, 'median')
        if patient_value > mean_val:
            return mean_val, 'reduce_to_median', patient_value - mean_val
        else:
            return mean_val, 'increase_to_median', patient_value - mean_val
    
    if feature == 'prior_admissions_12mo':
        if patient_value > 0:
            return 0, 'remove', -patient_value
        else:
            return 1, 'add', 1
        
    if feature == 'comorbidity_score':
        val = get_reference('comorbidity_score', patient_row, 'median')
        if patient_value > val:
            return val, 'reduce_to_median', patient_value - val
        else:
            return val, 'increase_to_median', patient_value - val
    
    if feature.startswith('icd_') or feature.startswith('ccsr_'):
        if patient_value == 1:
            return 0, 'remove', -1
        else:
            return 1, 'add', 1
    
    if feature == 'num_diagnoses':
        val = get_reference('num_diagnoses', patient_row, 'median')
        if patient_value > val:
            return val, 'reduce_to_median', patient_value - val
        else:
            return val, 'increase_to_median', patient_value - val
    
    if feature == 'num_chronic':
        val = get_reference('num_chronic', patient_row, 'median')
        if patient_value > val:
            return val, 'reduce_to_median', patient_value - val
        else:
            return val, 'increase_to_median', patient_value - val

In [7]:
all_results = []
for patient_id in tqdm(test_patients, desc="Patients"):
    patient_data = X[X['subject_id'] == patient_id].copy()
    
    for hadm_id in patient_data['hadm_id'].unique():
        hadm_data = patient_data[patient_data['hadm_id'] == hadm_id].copy()
        hadm_data_no_ids = hadm_data.drop(['subject_id', 'hadm_id', 'age_group'], axis=1)
        
        risk_actual = calibrated.predict_proba(hadm_data_no_ids)[:, 1]
        n_days = len(hadm_data_no_ids)

        for feature in features_to_analyze:
            hadm_counter = hadm_data_no_ids.copy()
            directions = []
            deviations = []
            feature_values = []
            counter_values = []
            for day_idx in range(n_days):
                patient_row = hadm_data.iloc[day_idx]
                current_value = patient_row[feature]
                counter_value, direction, deviation = get_counterfactual_value(
                    feature,
                    current_value,
                    patient_row
                )
                hadm_counter.iat[
                    day_idx,
                    hadm_counter.columns.get_loc(feature)
                ] = counter_value

                directions.append(direction)
                deviations.append(deviation)
                feature_values.append(current_value)
                counter_values.append(counter_value)

            risk_counter = calibrated.predict_proba(hadm_counter)[:,1]
            delta = risk_actual - risk_counter
            
            all_results.append({
                "subject_id": patient_id,
                "hadm_id": hadm_id,
                "feature": feature,
                "risk_actual": risk_actual,
                "risk_counterfactual": risk_counter,
                "delta": delta,
                "los": n_days,
                "feature_values": feature_values,
                "counterfactual_values": counter_values,
                "directions": directions,
                "deviations": deviations
            })

results_df = pd.DataFrame(all_results)
results_df.to_csv('deltas_full_results.csv', index=False)

Patients: 100%|██████████| 35915/35915 [1:31:59<00:00,  6.51it/s]  


In [59]:
import pandas as pd
results_df = pd.read_csv('deltas_full_results.csv')
results_df.head(5)

,subject_id,hadm_id,feature,risk_actual,risk_counterfactual,delta,los,feature_values,counterfactual_values,directions,deviations
0,19599026,25985099,icd_I10,[0.06145692 0.06875022 0.06221405],[0.06097641 0.06875022 0.06221405],[0.00048051 0. 0. ],3,"[np.float64(0.0), np.float64(0.0), np.float64(...","[1, 1, 1]","['add', 'add', 'add']","[1, 1, 1]"
1,19599026,25985099,icd_E785,[0.06145692 0.06875022 0.06221405],[0.06054453 0.06705463 0.06091155],[0.00091239 0.0016956 0.00130251],3,"[np.float64(0.0), np.float64(0.0), np.float64(...","[1, 1, 1]","['add', 'add', 'add']","[1, 1, 1]"
2,19599026,25985099,icd_K219,[0.06145692 0.06875022 0.06221405],[0.06149943 0.06930766 0.06172562],[-4.25060351e-05 -5.57437010e-04 4.88436903e-04],3,"[np.float64(0.0), np.float64(0.0), np.float64(...","[1, 1, 1]","['add', 'add', 'add']","[1, 1, 1]"
3,19599026,25985099,icd_Z87891,[0.06145692 0.06875022 0.06221405],[0.06101509 0.07138391 0.06472844],[ 0.00044183 -0.00263368 -0.00251439],3,"[np.float64(0.0), np.float64(0.0), np.float64(...","[1, 1, 1]","['add', 'add', 'add']","[1, 1, 1]"
4,19599026,25985099,icd_I2510,[0.06145692 0.06875022 0.06221405],[0.06378452 0.0722755 0.06522226],[-0.0023276 -0.00352528 -0.0030082 ],3,"[np.float64(0.0), np.float64(0.0), np.float64(...","[1, 1, 1]","['add', 'add', 'add']","[1, 1, 1]"


In [72]:
import ast
import re
from collections import Counter
import re
import numpy as np

def parse_numeric_array(x):

    if isinstance(x, np.ndarray):
        return x.astype(float)

    if isinstance(x, list):
        return np.asarray(x, dtype=float)

    if not isinstance(x, str):
        return np.asarray([x], dtype=float)

    s = x
    s = re.sub(r'np\.float64\((.*?)\)', r'\1', s)
    s = re.sub(r'np\.int64\((.*?)\)', r'\1', s)
    s = s.replace("nan", "np.nan")

    try:
        arr = np.asarray(eval(s, {"np": np}), dtype=float)
    except Exception:
        arr = np.fromstring(s.strip("[]"), sep=" ")

    return arr

def mean(x):
    arr = parse_numeric_array(x)
    return np.nanmean(arr)

def mean_abs(x):
    arr = parse_numeric_array(x)
    return np.nanmean(np.abs(arr))

def parse_list(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

def direction_mode(x):
    vals = parse_list(x)
    return Counter(vals).most_common(1)[0][0]

def direction_consistency(x):
    vals = parse_list(x)
    c = Counter(vals)
    return c.most_common(1)[0][1] / len(vals)

In [73]:
results_df["mean_current_value"] = results_df["feature_values"].apply(mean)
results_df["mean_counterfactual_value"] = results_df["counterfactual_values"].apply(mean)

results_df["delta_mean_abs"] = results_df["delta"].apply(mean_abs)
results_df["delta_mean"] = results_df["delta"].apply(mean)

results_df["risk_actual_mean"] = results_df["risk_actual"].apply(mean)
results_df["risk_counterfactual_mean"] = results_df["risk_counterfactual"].apply(mean)

results_df["direction_mode"] = results_df["directions"].apply(direction_mode)
results_df["direction_consistency"] = results_df["directions"].apply(direction_consistency)

In [75]:
feature_importance = (
    results_df
    .groupby("feature")
    .agg(
        mean_delta=("delta_mean", "mean"),
        mean_abs_delta=("delta_mean_abs", "mean"),
        std_delta=("delta_mean", "std"),
        std_abs=("delta_mean_abs", "std"),
        direction=("direction_mode", lambda x: x.mode().iat[0]),
        direction_consistency=("direction_consistency", "mean"),
    )
    .sort_values("mean_abs_delta", ascending=False)
)
feature_importance

,mean_delta,mean_abs_delta,std_delta,std_abs,direction,direction_consistency
feature,,,,,,
prior_admissions_12mo,0.029361,0.062165,0.092736,0.074838,add,1.000000
comorbidity_score,0.000790,0.017137,0.028778,0.023159,increase_to_median,1.000000
num_medications_daily,-0.005110,0.011885,0.018047,0.015173,remove,0.954965
ccsr_CIR019,0.006344,0.010229,0.012779,0.009953,add,1.000000
lab_51277_daily,0.009667,0.009864,0.018758,0.018660,normal,0.932526
ccsr_BLD003,-0.006055,0.008979,0.009891,0.007401,add,1.000000
ccsr_EXT027,0.005416,0.008059,0.012439,0.010928,add,1.000000
num_diagnoses,-0.006323,0.007655,0.009500,0.008493,increase_to_median,1.000000
num_chronic,-0.005435,0.007372,0.010010,0.008724,increase_to_median,1.000000


In [76]:
columns_to_save = ['subject_id', 'hadm_id', 'feature', 'los', \
                   'risk_actual_mean', 'risk_counterfactual_mean', \
                   'mean_current_value', 'mean_counterfactual_value',\
                    'delta_mean', 'delta_mean_abs', 'direction_mode', 'direction_consistency']
results_df[columns_to_save].to_csv('total_deltas.csv', index=False)

In [78]:
feature_importance = feature_importance.reset_index()
feature_importance.to_csv('delta_summary.csv', index=False)

In [79]:
print("\nFactors that increase risk (Δ > 0):")
positive = feature_importance[feature_importance['mean_delta'] > 0].sort_values('mean_delta', ascending=False).head(10)
for _, row in positive.iterrows():
    print(f"  {row['feature']}: Δ = {row['mean_delta']:+.4f} ± {row['std_delta']:.4f} [{row['direction']}]")

print("\nFactors that decrease risk (Δ < 0):")
negative = feature_importance[feature_importance['mean_delta'] < 0].sort_values('mean_delta', ascending=True).head(10)
for _, row in negative.iterrows():
    print(f"  {row['feature']}: Δ = {row['mean_delta']:+.4f} ± {row['std_delta']:.4f} [{row['direction']}]")


Factors that increase risk (Δ > 0):
  prior_admissions_12mo: Δ = +0.0294 ± 0.0927 [add]
  lab_51277_daily: Δ = +0.0097 ± 0.0188 [normal]
  ccsr_CIR019: Δ = +0.0063 ± 0.0128 [add]
  lab_51279_daily: Δ = +0.0063 ± 0.0115 [normal]
  ccsr_EXT027: Δ = +0.0054 ± 0.0124 [add]
  lab_51221_daily: Δ = +0.0046 ± 0.0083 [low]
  ccsr_END009: Δ = +0.0043 ± 0.0076 [add]
  ccsr_END003: Δ = +0.0027 ± 0.0069 [add]
  lab_51222_daily: Δ = +0.0020 ± 0.0039 [normal]
  icd_E785: Δ = +0.0020 ± 0.0072 [add]

Factors that decrease risk (Δ < 0):
  num_diagnoses: Δ = -0.0063 ± 0.0095 [increase_to_median]
  ccsr_BLD003: Δ = -0.0061 ± 0.0099 [add]
  num_chronic: Δ = -0.0054 ± 0.0100 [increase_to_median]
  num_medications_daily: Δ = -0.0051 ± 0.0180 [remove]
  ccsr_END011: Δ = -0.0037 ± 0.0080 [add]
  ccsr_CIR007: Δ = -0.0036 ± 0.0131 [add]
  icd_F329: Δ = -0.0023 ± 0.0097 [add]
  icd_I4891: Δ = -0.0013 ± 0.0034 [add]
  icd_N179: Δ = -0.0012 ± 0.0029 [add]
  icd_Z7901: Δ = -0.0012 ± 0.0039 [add]
